<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Models

In [2]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

fatal: destination path 'DS5001-Final-Project' already exists and is not an empty directory.


In [3]:
!wget -O TFIDF_L2.csv "https://virginia.box.com/shared/static/959w70gbj2ckxew7clguvy3updzfn3t6"
!wget -O LIB.csv "https://virginia.box.com/shared/static/fhzudg34je9xls5bfcbi4xdnaiek74rj.csv"

--2026-04-13 15:08:00--  https://virginia.box.com/shared/static/959w70gbj2ckxew7clguvy3updzfn3t6
Resolving virginia.box.com (virginia.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.box.com (virginia.box.com)|74.112.186.157|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /public/static/959w70gbj2ckxew7clguvy3updzfn3t6 [following]
--2026-04-13 15:08:00--  https://virginia.box.com/public/static/959w70gbj2ckxew7clguvy3updzfn3t6
Reusing existing connection to virginia.box.com:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://virginia.app.box.com/public/static/959w70gbj2ckxew7clguvy3updzfn3t6 [following]
--2026-04-13 15:08:01--  https://virginia.app.box.com/public/static/959w70gbj2ckxew7clguvy3updzfn3t6
Resolving virginia.app.box.com (virginia.app.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.app.box.com (virginia.app.box.com)|74.112.186.157|:443... connected.
HTTP r

In [4]:
import pandas as pd
TFIDF_L2 = pd.read_csv('TFIDF_L2.csv', delimiter='|', index_col=[0,1,2])
LIB = pd.read_csv('LIB.csv', delimiter='|', index_col=0)

## PCA Components

In [5]:
from sklearn.decomposition import PCA
import numpy as np

pca_engine = PCA(n_components=5)
pca_engine.fit(TFIDF_L2)

COMPS = pd.DataFrame(pca_engine.components_.T * np.sqrt(pca_engine.explained_variance_))
COMPS.columns = ["PC{}".format(i) for i in COMPS.columns]
COMPS.index = TFIDF_L2.columns
COMPS.index.name = 'term_str'
COMPS.head()

,PC0,PC1,PC2,PC3,PC4
term_str,,,,,
a,-0.002289,-0.002043,-0.001101,-0.000119,-0.000969
all,-0.002790,-0.004950,-0.000796,0.002500,0.000028
and,-0.002442,-0.003982,-0.002092,0.001712,-0.002339
around,-0.001855,-0.002173,-0.001955,0.000988,-0.000920
at,-0.001911,-0.003871,-0.002408,0.001567,-0.001959


In [6]:
print("Top 5 positive terms for PC0:")
print(COMPS['PC0'].nlargest(5))

print("\nTop 5 negative terms for PC1:")
print(COMPS['PC1'].nsmallest(5))

Top 5 positive terms for PC0:
term_str
released    0.119626
song        0.075531
lyrics      0.052784
check       0.050403
yet         0.050178
Name: PC0, dtype: float64

Top 5 negative terms for PC1:
term_str
you    -0.010204
love   -0.009202
let    -0.008622
dont   -0.008612
need   -0.008499
Name: PC1, dtype: float64


In [7]:
COMPS.to_csv('/content/DS5001-Final-Project/COMPS.csv', sep='|', index=True)

## PCA DCM

In [8]:
TFIDF_L2.index = (
    TFIDF_L2.index.get_level_values('Artist') + ' — ' +
    TFIDF_L2.index.get_level_values('Title') + ' (' +
    TFIDF_L2.index.get_level_values('Album') + ')'
)
TFIDF_L2.index.name = 'track_id'

In [9]:
DCM = pd.DataFrame(pca_engine.fit_transform(TFIDF_L2), index=TFIDF_L2.index)
DCM.columns = ['PC{}'.format(i) for i in DCM.columns]
DCM.head()

,PC0,PC1,PC2,PC3,PC4
track_id,,,,,
Ariana Grande — Brand New You (13 (Original Broadway Cast Recording)),-0.032332,0.071247,-0.013300,-0.065710,-0.004599
Ariana Grande — Ariana Grande Tour Guide (Ariana Grande),-0.019869,0.118616,0.003782,-0.092360,0.008104
Ariana Grande — I’m Every Woman/Vogue (Ariana Grande),-0.048779,-0.003187,0.050625,-0.004128,-0.004922
Ariana Grande — Leave Me Lonely (Reprise) (Ariana Grande),-0.024795,0.087238,0.011210,-0.065125,0.032759
Ariana Grande — Love Me Harder/breathin (Ariana Grande),-0.029093,0.081931,0.005862,-0.063612,0.017884


In [10]:
DCM.to_csv('/content/DS5001-Final-Project/DCM.csv', sep='|', index=True)

## PCA Loadings

## PCA Visualization 1

In [11]:
import plotly_express as px
import plotly.io as pio
pio.renderers.default = 'colab'

def vis_pcs(DCM, a, b, label='Artist'):
    LIB_reduced = LIB.loc[DCM.index]
    fig = px.scatter(DCM, f"PC{a}", f"PC{b}",
                    color=LIB_reduced[label],
                    hover_name=LIB_reduced.index,
                    size=LIB_reduced['doc_length_words'],
                    marginal_x='box',
                    height=800,
                    title=f"PCA — PC{a} vs PC{b} colored by {label}")
    fig.update_traces(marker=dict(line=dict(width=0)))
    fig.update_layout(
        xaxis=dict(showgrid=True, gridcolor='lightgrey'),
        yaxis=dict(showgrid=True, gridcolor='lightgrey')
    )
    return fig

def vis_loadings(COMPS, a=0, b=1, hover_name='term_str'):
    fig = px.scatter(COMPS.reset_index(), f"PC{a}", f"PC{b}",
                      text='term_str',
                      marginal_x='box',
                      height=800,
                      title=f"PCA Loadings — PC{a} vs PC{b}")
    fig.update_traces(marker=dict(line=dict(width=0)))
    fig.update_layout(
        xaxis=dict(showgrid=True, gridcolor='lightgrey'),
        yaxis=dict(showgrid=True, gridcolor='lightgrey')
    )
    return fig

In [21]:
import kaleido
fig1 = vis_pcs(DCM, 0, 1)
pio.write_image(fig1, "/content/DS5001-Final-Project/images/PCA_components_PC0_PC1.png", format="png", scale=2, engine="kaleido")
fig1

In [22]:
fig2 = vis_loadings(COMPS, 0, 1)
pio.write_image(fig2, "/content/DS5001-Final-Project/images/PCA_loadings_PC0_PC1.png", format="png", scale=2, engine="kaleido")
fig2

In [14]:
print("Positive pole (right side):")
print(COMPS['PC0'].nlargest(10))

print("\nNegative pole (left side):")
print(COMPS['PC0'].nsmallest(10))

Positive pole (right side):
term_str
released    0.119626
song        0.075531
lyrics      0.052784
check       0.050403
yet         0.050178
once        0.046168
has         0.040064
please      0.038928
been        0.020751
back        0.018920
Name: PC0, dtype: float64

Negative pole (left side):
term_str
love   -0.004754
oh     -0.004733
yeah   -0.004613
you    -0.004581
baby   -0.004212
want   -0.003929
i      -0.003788
dont   -0.003644
me     -0.003475
we     -0.003448
Name: PC0, dtype: float64


##PCA Visualization 2

In [23]:
fig3 = vis_pcs(DCM, 1, 2)
pio.write_image(fig3, "/content/DS5001-Final-Project/images/PCA_components_PC1_PC2.png", format="png", scale=2, engine="kaleido")
fig3

In [24]:
fig4 = vis_loadings(COMPS, 1, 2)
pio.write_image(fig4, "/content/DS5001-Final-Project/images/PCA_loadings_PC1_PC2.png", format="png", scale=2, engine="kaleido")
fig4

In [17]:
print("Positive pole (right side):")
print(COMPS['PC1'].nlargest(10))

print("\nNegative pole (left side):")
print(COMPS['PC1'].nsmallest(10))

Positive pole (right side):
term_str
la           0.045463
edge         0.005381
christmas    0.004334
chris        0.003527
snippet      0.003429
martin       0.003283
light        0.003094
fa           0.003039
05           0.002851
gaga         0.002839
Name: PC1, dtype: float64

Negative pole (left side):
term_str
you     -0.010204
love    -0.009202
let     -0.008622
dont    -0.008612
need    -0.008499
never   -0.008469
baby    -0.007927
if      -0.007777
ill     -0.007598
back    -0.007238
Name: PC1, dtype: float64


##LDA Topic